# Lab 07 · Enforce cost budgets with per-caller overrides

Publish an alias with a dollar budget for each caller, spend it with a short series of requests, watch the gateway
answer `403` once the estimated spend runs out, and prove that a key with an override keeps working. This notebook
is the Python track of the [lab guide](README.md): it deploys the same [infra/budget-alias.bicep](infra/budget-alias.bicep)
as the Bicep + azd track and sends the same requests as [infra/trigger-cost-limit.sh](infra/trigger-cost-limit.sh).

The lab guide sets a budget of $5 per day. Here the budget is $0.001 per hour, so about three calls spend it, and the
block lifts at the start of the next hour instead of the next day. The template doesn't touch the model or alias
that your clients already call: it adds the alias `gpt-5-mini-budget`, which targets the same backend model.

**Check the model → Deploy the budgeted alias → Spend until 403 → Chart the spend → Inspect 403 → Prove the override → Clean up**

## Architecture at a glance

| Component | What it does |
| --- | --- |
| Backend model `foundry-models/gpt-5-mini` | The model from notebook 02. Its own limits stay as they are. |
| Alias `gpt-5-mini-budget` | Targets the same model and carries a `costLimit` policy: $0.001 per hour, counted per `identity`, so every runtime key gets its own budget. |
| Runtime key `app-dev` | The caller with the default budget, which runs out. |
| Runtime key `support-bot` | A caller with an override: its own budget of $50 per hour. |
| [infra/budget-alias.bicep](infra/budget-alias.bicep) | Declares the budgeted alias, both keys, and the override. |
| [../shared/utils.py](../shared/utils.py) | Helpers that every lab notebook shares: settings, your `az login` sign-in, Bicep deployments, keys, and retries. |

![A cost limit can apply to one model or to the whole gateway, with overrides for specific callers](../assets/diagrams/policy-scopes.svg)

## Prerequisites

- An Azure subscription where you can create resources (Contributor on the resource group) and assign roles
  (Owner, User Access Administrator, or Role Based Access Control Administrator).
- The [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli), signed in with `az login`. The notebook
  deploys Bicep with `az deployment group create`, and the CLI installs Bicep the first time it needs it.
- Python 3.10 or later with Jupyter, for example VS Code with the Python and Jupyter extensions.
- [01-provision-gateway.ipynb](../01-provision-gateway/01-provision-gateway.ipynb) and
  [02-first-model-call.ipynb](../02-first-model-call/02-first-model-call.ipynb) have run, so the gateway publishes
  the `gpt-5-mini` model.

Run the cells in order, or click **Run All**. Each step is idempotent: rerun any cell and it updates the same resources.

In [ ]:
%pip install -q --upgrade azure-identity requests openai matplotlib

### 0. Initialize the lab

The defaults match notebooks 01 and 02, so `utils.Lab` finds the gateway without edits. The budgeted alias targets
the backend model `gpt-5-mini` under the `foundry-models` provider that notebook 02 published. Amounts are in US
dollars and are strings, so that they can hold decimals such as `0.001`.

In [ ]:
import re
import sys
from IPython.display import display

sys.path.insert(0, "../shared")   # helpers and brand styles that every lab notebook shares
import brand
import utils

lab = utils.Lab("07-cost-budgets")
provider = "foundry-models"   # model provider from notebook 02
model    = "gpt-5-mini"       # backend model that the budgeted alias targets
budget   = "0.001"            # default budget for each caller, in US dollars: about three calls to gpt-5-mini
period   = "hour"             # hour, day, week, month, or year: budgets renew at the start of each calendar period
override = "50"               # support-bot's own budget, in US dollars

### 1. Check the model

The cell confirms that notebook 02 published the model, and notes which of the `app-dev` and `support-bot` keys
don't exist yet. The template creates them, and the clean-up step deletes only those.

In [ ]:
lab.require(f"{lab.workspace_id}/modelProviders/{provider}/models/{model}", "Run 02-first-model-call.ipynb first.")
display(brand.note(f"Model {provider}/{model} exists", "ok"))

created_keys = [name for name in ("app-dev", "support-bot") if lab.arm("GET", f"{lab.gateway_id}/apiKeys/{name}") is None]
display(brand.note("Keys that the template creates: " + (", ".join(created_keys) or "none, both exist"), "brand"))

### 2. Deploy the budgeted alias

[infra/budget-alias.bicep](infra/budget-alias.bicep) creates the alias `<model>-budget` with a `policies` array that
holds one cost limit, and the two runtime keys. You set the budget with the template's parameters:

- `budgetAmount` is the default budget for each caller, in US dollars, and `period` is how often it renews.
- `overrideAmount` is the budget of the `support-bot` key. Overrides match an API key's resource ID or, with
  `entraObjectId`, a Microsoft Entra identity, and the first match wins.
- The counter key is always `identity` (**Caller identity** in the portal), because overrides need a counter for
  each caller.

The gateway estimates the cost of each call from the model's token prices. The policy's `id` is stable, so when you
change the amount and deploy again, the spend so far still counts.

In [ ]:
out = lab.deploy("infra/budget-alias.bicep", gatewayName=lab.gateway_name, providerName=provider, modelName=model,
                 budgetAmount=budget, period=period, overrideAmount=override)
alias = out["aliasName"]

policies = lab.arm("GET", f"{lab.workspace_id}/aliases/{alias}")["properties"]["policies"]
display(brand.policies(alias, policies))
cost_limit = next(p for p in policies if p["type"] == "costLimit")
assert float(cost_limit["amount"]) == float(budget) and cost_limit["period"] == period
assert cost_limit["counterKey"] == ["identity"] and len(cost_limit.get("overrides", [])) == 1

### 3. Spend the budget until 403

Send requests with the `app-dev` key, which each cost about $0.0004, until two are blocked. The gateway reports
the cost in response headers that start with `x-alias-cost-per-hour-`:

- An answered call carries `…-consumed`, its estimated cost.
- A blocked call carries `…-remaining`, which is `0`, and `…-retry-after`, the seconds until the budget renews at
  the start of the next hour in UTC.

On a cost limit attached to the model instead of an alias, the headers start with `x-cost-`. The cell reads both.

The gateway checks the budget before it forwards a request, and adds the cost once the response is back, so the call
that takes the spend past the budget still succeeds. The client sets `max_retries=0` so that you see each status. A
new alias and its policy can take a minute to apply, so the cell first waits until the alias answers, and repeats
the series if no request is blocked. The spend so far still counts.

In [ ]:
from openai import OpenAI, PermissionDeniedError, APIStatusError

PROMPT   = "Write a 150-word note to a finance team explaining why per-app AI budgets help."
COST     = re.compile(r"^x-(?:alias-)?cost-per-\w+-(consumed|remaining|retry-after)$")
base_url = f"{lab.gateway_url}/default/models/openai/v1"

def cost_headers(headers):
    """Return the cost headers of a response, keyed by consumed, remaining, and retry-after."""
    return {match[1]: value for name, value in headers.items() if (match := COST.match(name.lower()))}

def send(client, prompt=PROMPT, max_completion_tokens=250):
    """Send one chat request. Return the status, response headers, body text, and estimated cost."""
    try:
        raw = client.chat.completions.with_raw_response.create(
            model=alias, max_completion_tokens=max_completion_tokens, reasoning_effort="none",
            messages=[{"role": "user", "content": prompt}])
        status, headers, body = raw.status_code, raw.headers, raw.text
    except APIStatusError as error:
        status, headers, body = error.status_code, error.response.headers, error.response.text
    consumed = cost_headers(headers).get("consumed")
    return status, headers, body, float(consumed) if status == 200 and consumed else None

def spend(client, attempt=1, count=20):
    """Send requests until two are blocked. A live table shows each status and the spend against the budget."""
    board = brand.Board(f"Spend against {alias}", ["Request", "Status", "Cost", "Retry after (s)"],
                        sub=f"app-dev · attempt {attempt}", budget=float(budget), unit="USD", stop=403,
                        budget_label=f"Spent in this run / budget per {period}")
    results = []
    for i in range(1, count + 1):
        status, headers, body, cost = send(client)
        found = cost_headers(headers)
        board.add([i, brand.status(status), brand.usd(cost), found.get("retry-after", "-")], status, cost)
        results.append((status, headers, body, cost))
        if sum(r[0] == 403 for r in results) == 2:
            break
    return results

app_dev_key, _ = lab.api_key(out["defaultKeyName"])
display(brand.key(out["defaultKeyName"], app_dev_key))
client = OpenAI(base_url=base_url, api_key=app_dev_key, max_retries=0)

try:
    utils.retry(lambda: client.chat.completions.create(   # wait until the new alias and key answer
        model=alias, max_completion_tokens=16, reasoning_effort="none",
        messages=[{"role": "user", "content": "ping"}]), statuses=(401, 404))
except PermissionDeniedError:
    display(brand.note(f"app-dev already spent its budget this {period}, so the requests are blocked right away.", "warn"))

for attempt in range(1, 4):
    results = spend(client, attempt=attempt)
    blocked = [r for r in results if r[0] == 403]
    if blocked:
        break
    display(brand.note("No 403 yet. The policy may still be applying.", "warn"))
    brand.wait(30, "before the next attempt")

assert blocked, "Expected at least one 403."
assert "cost quota" in blocked[-1][2].lower(), "A blocked response should say that the LLM cost quota is exceeded."

### 4. Chart the spend

Plot the last series. Each bar is the estimated cost of one request, and the line is the running total of this
series against the dashed budget. The call that takes the total past the budget still succeeds, and the calls after
it get `403`. Blocked calls never reach the model and cost nothing, so the line stays flat. The line counts only
this run: if `app-dev` spent part of its budget earlier this hour, the first `403` comes before the line reaches the
budget. The chart follows your notebook's light or dark theme, and **Show the data** lists the numbers behind it.

In [ ]:
display(brand.usage_chart("Estimated cost per request against the budget", [(r[0], r[3]) for r in results],
                          limit=float(budget), limit_label=f"Budget: {brand.usd(budget)} per {period}", unit="USD",
                          sub=f"{alias} · app-dev"))

### 5. Inspect a blocked response

Look at the status, headers, and body of one blocked call. The body says `LLM cost quota is exceeded` and how long
to wait, and `…-retry-after` gives the same wait in seconds. Unlike a token limit, which frees up as its window
rolls, the caller stays blocked until the next period starts or you raise the budget. The gateway rejected the
request itself: the backend didn't receive it.

In [ ]:
status, headers, body, _ = blocked[-1]
display(brand.http(status, headers, body, "A blocked response", sub=alias))

answered = [r for r in results if r[0] == 200]
if answered:
    display(brand.info("Cost headers of the last answered request",
                       [(brand.mono(name), value) for name, value in answered[-1][1].items() if COST.match(name.lower())],
                       sub="app-dev"))

### 6. Prove the override and the per-caller counters

Call the alias with the `support-bot` key while `app-dev` is still blocked. `support-bot` has its own counter and an
override of $50 per hour, so its requests succeed. A new key can take a moment to work, so the cell retries `401`
with a cheap request first.

In [ ]:
support_key, _ = lab.api_key(out["overrideKeyName"])
display(brand.key(out["overrideKeyName"], support_key))
support_client = OpenAI(base_url=base_url, api_key=support_key, max_retries=0)
utils.retry(lambda: support_client.chat.completions.create(
    model=alias, max_completion_tokens=16, reasoning_effort="none",
    messages=[{"role": "user", "content": "ping"}]), statuses=(401,), wait=15)

rows = [(brand.mono("app-dev"), send(client)[0], 403, "Its budget is spent")]
for i in range(1, 4):
    status, _, _, cost = send(support_client)
    rows.append((brand.mono(f"support-bot #{i}"), status, 200, f"Override budget, cost {brand.usd(cost)}"))

display(brand.checks("One budget counter per key", rows, sub=alias))
assert rows[0][1] == 403 and all(row[1] == 200 for row in rows[1:])

### 7. Clean up resources

The budgeted alias blocks anyone who has spent its budget, so the cell deletes it. The model and its `gpt-5-mini`
alias were never changed. The cell deletes the `app-dev` and `support-bot` keys only if the template created them in
step 2.

To see the estimated spend by key and model in **Monitoring**, configure Application Insights first (lab 13).

In [ ]:
lab.delete(f"{lab.workspace_id}/aliases/{alias}")
for name in created_keys:
    lab.delete(f"{lab.gateway_id}/apiKeys/{name}")

## Troubleshooting

- **`403` on the first request**: `app-dev` spent its budget earlier this hour, for example in an earlier run.
  Budgets renew at the start of each period, so wait for the `…-retry-after` seconds. Editing the amount keeps the
  spend so far.
- **No `403` after several series**: a policy change takes up to a minute to reach the gateway. Check that the
  request `model` is the budgeted alias, and that
  `lab.arm("GET", f"{lab.workspace_id}/aliases/{alias}")["properties"]["policies"]` lists the cost limit.
- **`429` instead of `403`**: a token limit or the backend's own capacity blocked the call. See
  [Throttle tokens per caller with token limits](../06-token-limits/README.md).
- **`support-bot` gets `403`**: its override is missing. Check that the cost limit's `overrides` holds the
  `support-bot` key's resource ID, and that no **Gateway** scope cost limit is exhausted.
- **The Cost column shows `-` for answered requests**: the responses have no `…-consumed` header. Check that the
  request `model` is the budgeted alias, and that the alias carries the cost limit.

For the other errors in this lab, see [Troubleshooting](../TROUBLESHOOTING.md#lab-07-cost-budgets).

## Next steps

- Combine gateway-scoped budgets with other guardrails in [Set gateway-wide policy baselines](../09-gateway-baselines/README.md).
- See estimated spend by API key and model in [Monitor usage, cost, and traces](../13-monitoring/README.md).
- Build chargeback reports by team or identity in [Attribute tokens and cost with OpenTelemetry](../14-otel-chargeback/README.md).
- Docs: [Policies in AI Gateway](https://ai.gateway.azure.com/docs/policies).
- Read this lab on the labs site: [https://azure.github.io/aigateway/labs/07-cost-budgets/](https://azure.github.io/aigateway/labs/07-cost-budgets/).